In [1]:
import pandas as pd
df = pd.read_pickle("dataset/dataset_v3.pkl")
df.head()

,A1_age_group,A2_gender,A3_religion,A4_marital_status,A5_residence,A6_occupation,B1_family_type,B2_family_time_group,B3_family_relationship,B4_in_person_feeling,...,G6_panic_attacks,G7_behavior_type,G8_addictive_watching,H1_mental_state_change,H2_content_type_effect,H3_distraction_after_viral,H4_brain_pressure,H5_sleep_quality_change,H6_emotional_reels_focus_loss,H7_curiosity_driven_rewatch
0,21-25,Male,Islam,Married,Urban Area,Student,Single family,1-3 hours,Good,Somewhat Comfortable,...,Sometimes,Extrovert,Sometimes,Some Change,Negative Impact,Sometimes,Moderate Stress,Slight Change in Sleep Quality,Lose Focus Frequently,Sometimes Interested
1,26-35,Male,Islam,Unmarried,Urban Area,Service Holder,Joint Family,1-3 hours,Very Good,Neutral,...,Never,Introvert,Never,No Change,No Impact,Never,No Stress,Not Sure,Do Not Lose Focus,Rarely Interested
2,21-25,Female,Islam,Unmarried,Urban Area,Student,Single family,5-8 hours,Very Good,Somewhat Comfortable,...,Sometimes,Introvert,Always,Some Change,Not Sure,Sometimes,No Stress,Not Sure,Lose Focus Sometimes,More Interest in Watching Reels
3,21-25,Male,Islam,Married,Urban Area,Student,Joint Family,1-3 hours,Moderate,Very Comfortable,...,Never,Ambivert,Sometimes,Some Change,Not Sure,Never,No Stress,Not Sure,Do Not Lose Focus,Sometimes Interested
4,21-25,Female,Islam,Unmarried,Urban Area,Student,Single family,Up to 1 hour,Good,Neutral,...,Sometimes,Ambivert,Always,Some Change,Not Sure,Rarely,Low Stress,Slight Change in Sleep Quality,Lose Focus Rarely,Sometimes Interested


In [2]:
import numpy as np
from scipy.stats import fisher_exact, chi2_contingency, MonteCarloMethod
from scipy.stats.contingency import expected_freq

OUT = 'D2_daily_watch_time'
ORDER = ['Less than 1 hour', '1 hour', '2 hours', '3 hours', 'More than that']
N_RESAMPLES = 100_000  # same settings as 05_fisher_test.ipynb, so p-values match
SEED = 42
ALPHA = 0.05

df[OUT] = pd.Categorical(df[OUT], categories=ORDER, ordered=True)
preds = [c for c in df.columns if c != OUT]

# Fisher-Freeman-Halton when any cell has an expected count of 5 or less;
# chi-square otherwise
rows = []
for c in preds:
    ct = pd.crosstab(df[c], df[OUT])
    if (expected_freq(ct) <= 5).any():
        test = 'Fisher-Freeman-Halton'
        p = fisher_exact(ct.values, method=MonteCarloMethod(n_resamples=N_RESAMPLES, rng=np.random.default_rng(SEED))).pvalue
    else:
        test = 'Chi-square'
        p = chi2_contingency(ct)[1]
    rows.append((c, test, p))

tests = pd.DataFrame(rows, columns=['Variable', 'Test', 'p-value'])
tests['Significant'] = tests['p-value'] < ALPHA
# Show p-values as 4 decimals; very small ones as '< 0.0001' (numbers stay unchanged in `tests`)
fmt_p = lambda p: '< 0.0001' if p < 0.0001 else f'{p:.4f}'
tests.sort_values('p-value').reset_index(drop=True).assign(**{'p-value': lambda t: t['p-value'].map(fmt_p)})

,Variable,Test,p-value,Significant
0,C3_sitting_duration,Chi-square,< 0.0001,True
1,G8_addictive_watching,Chi-square,< 0.0001,True
2,H7_curiosity_driven_rewatch,Chi-square,< 0.0001,True
3,H6_emotional_reels_focus_loss,Chi-square,< 0.0001,True
4,F4_performance_affected,Fisher-Freeman-Halton,< 0.0001,True
5,F2_memory_reduced,Fisher-Freeman-Halton,< 0.0001,True
6,F1_attention_decreased,Fisher-Freeman-Halton,< 0.0001,True
7,C4_physical_problems,Chi-square,< 0.0001,True
8,H3_distraction_after_viral,Chi-square,< 0.0001,True
9,G3_social_comparison,Chi-square,< 0.0001,True


In [3]:
# Separate significant and non-significant variables
sig_cols = tests.loc[tests['Significant'], 'Variable'].tolist()
non_sig_cols = tests.loc[~tests['Significant'], 'Variable'].tolist()

print(f"Significant with {OUT} (p < {ALPHA}): {len(sig_cols)}")
for c in sig_cols:
    print('  ', c)
print()
print(f"Not significant: {len(non_sig_cols)}")
for c in non_sig_cols:
    print('  ', c)

Significant with D2_daily_watch_time (p < 0.05): 28
   A4_marital_status
   B2_family_time_group
   B3_family_relationship
   B4_in_person_feeling
   C2_weight_group
   C3_sitting_duration
   C4_physical_problems
   D3_peak_watch_time
   E2_sleep_trouble
   E3_exercise_involvement
   F1_attention_decreased
   F2_memory_reduced
   F3_learning_interest_decreased
   F4_performance_affected
   G1_mood_swings
   G2_fomo_anxiety
   G3_social_comparison
   G4_social_isolation
   G5_mental_stress
   G6_panic_attacks
   G8_addictive_watching
   H1_mental_state_change
   H2_content_type_effect
   H3_distraction_after_viral
   H4_brain_pressure
   H5_sleep_quality_change
   H6_emotional_reels_focus_loss
   H7_curiosity_driven_rewatch

Not significant: 12
   A1_age_group
   A2_gender
   A3_religion
   A5_residence
   A6_occupation
   B1_family_type
   C1_height_group
   D1_years_group
   D4_multitasking_while_watching
   E1_sleep_hours
   E4_substance_use
   G7_behavior_type


In [4]:
# Keep the outcome + significant variables only
df_sig = df[[OUT] + sig_cols]
df_sig.to_pickle("dataset/dataset_v4_sig.pkl")
print(f"Saved dataset/dataset_v4_sig.pkl with shape {df_sig.shape}")

Saved dataset/dataset_v4_sig.pkl with shape (503, 29)
